In [ ]:
# @title Install required dependencies
!pip install -q "crewai[tools]>=1.8.0" python-dotenv ddgs pydantic pandas pypdf


In [ ]:
import os
from pydantic import BaseModel
from google.colab import userdata
from crewai import LLM
from crewai.flow.flow import Flow, listen, router, start

class SupportState(BaseModel):
    ticket_text: str = ""
    customer_name: str = ""
    category: str = ""
    priority: str = "medium"
    resolution: str = ""

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    llm = LLM(
        model="google/gemini-3.1-flash-lite-preview",
        api_key=userdata.get("GEMINI_API_KEY"),
        temperature=0.3,
        max_tokens=800,
    )

    class CustomerSupportFlow(Flow[SupportState]):
        @start()
        def receive_ticket(self):
            ticket = input("Enter support ticket inquiry [or press Enter for default]: ").strip()
            if not ticket:
                ticket = "I was charged twice for subscription invoice #9823 on my Visa card."
            self.state.ticket_text = ticket
            self.state.customer_name = "Valued Customer"

        @listen(receive_ticket)
        def categorize_ticket(self):
            prompt = (
                f"Classify this support ticket into exactly one word: billing, technical, account, or other.\n"
                f"Ticket: {self.state.ticket_text}\n"
                f"Respond with ONLY the category word in lowercase."
            )
            cat = llm.call(prompt).strip().lower()
            if "billing" in cat:
                self.state.category = "billing"
                self.state.priority = "high"
            elif "tech" in cat:
                self.state.category = "technical"
                self.state.priority = "high"
            elif "account" in cat:
                self.state.category = "account"
                self.state.priority = "medium"
            else:
                self.state.category = "other"
                self.state.priority = "low"

        @router(categorize_ticket)
        def route_by_category(self):
            return self.state.category

        @listen("billing")
        def handle_billing(self):
            prompt = f"Draft a professional billing support response for: {self.state.ticket_text}. State that refund review takes 24 hours."
            self.state.resolution = llm.call(prompt)

        @listen("technical")
        def handle_technical(self):
            prompt = f"Draft an empathetic technical support response for: {self.state.ticket_text}. Request reproduction steps."
            self.state.resolution = llm.call(prompt)

        @listen("account")
        def handle_account(self):
            prompt = f"Draft an account security verification response for: {self.state.ticket_text}. Provide secure password reset steps."
            self.state.resolution = llm.call(prompt)

        @listen("other")
        def handle_other(self):
            prompt = f"Draft a polite customer support response acknowledging: {self.state.ticket_text}."
            self.state.resolution = llm.call(prompt)

    flow = CustomerSupportFlow()
    flow.kickoff()
    print(f"\n--- Ticket Classification: {flow.state.category.upper()} (Priority: {flow.state.priority}) ---")
    print("\nResolution:\n", flow.state.resolution)

except Exception as e:
    print("Error:", e)
